In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

DATA_FILE = DATA_DIR / "dataSet_withFeatures.csv"
LABEL_FILE = DATA_DIR / "dataSet_withFeatures_labels.csv"

data = pd.read_csv(DATA_FILE, header=None)
labels = pd.read_csv(LABEL_FILE, header=None).iloc[:, 0]

print("Data:", data.shape)
print("Labels:", labels.shape)

Data: (5990, 448)
Labels: (5990,)


In [3]:
X = data.iloc[:, :447].copy()
y = labels.astype(str).str.strip().copy()

print("X:", X.shape)
print("y:", y.shape)

X: (5990, 447)
y: (5990,)


In [4]:
X_tactile = X.iloc[:, :414].to_numpy(dtype=np.float32)
X_features = X.iloc[:, 414:447].to_numpy(dtype=np.float32)

print("Tactile:", X_tactile.shape)
print("Features:", X_features.shape)

Tactile: (5990, 414)
Features: (5990, 33)


In [6]:
label_map = {
    "asphalt": 0,
    "cork": 1,
    "grass": 2,
    "gravel": 3,
    "lab": 4,
    "laminate_wood": 5,
    "pebble": 6,
    "sand": 7
}
y_encoded = y.map(label_map)

print(y_encoded.head())
print("\nMissing encoded labels:", y_encoded.isna().sum())

0    0
1    0
2    0
3    0
4    0
Name: 0, dtype: int64

Missing encoded labels: 0


In [7]:
print(
    pd.DataFrame({
        "terrain": y,
        "encoded": y_encoded
    }).drop_duplicates().sort_values("encoded")
)

            terrain  encoded
0           asphalt        0
404            cork        1
1136          grass        2
1837         gravel        3
2324            lab        4
2725  laminate_wood        5
3095         pebble        6
3820           sand        7


In [8]:
X_tactile_train, X_tactile_temp, \
X_features_train, X_features_temp, \
y_train, y_temp = train_test_split(
    X_tactile,
    X_features,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

In [9]:
X_tactile_val, X_tactile_test, \
X_features_val, X_features_test, \
y_val, y_test = train_test_split(
    X_tactile_temp,
    X_features_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [10]:
print("Training:", len(y_train))
print("Validation:", len(y_val))
print("Test:", len(y_test))

Training: 4193
Validation: 898
Test: 899


In [11]:
def show_distribution(y, name):
    counts = pd.Series(y).value_counts().sort_index()

    print(f"\n{name}")
    print(counts)

show_distribution(y_train, "Training")
show_distribution(y_val, "Validation")
show_distribution(y_test, "Test")


Training
0
0    559
1    512
2    491
3    509
4    542
5    540
6    508
7    532
Name: count, dtype: int64

Validation
0
0    119
1    110
2    105
3    109
4    116
5    116
6    109
7    114
Name: count, dtype: int64

Test
0
0    120
1    110
2    105
3    109
4    117
5    116
6    108
7    114
Name: count, dtype: int64


In [12]:
X_tactile_train = X_tactile_train.reshape(-1, 6, 69)
X_tactile_val = X_tactile_val.reshape(-1, 6, 69)
X_tactile_test = X_tactile_test.reshape(-1, 6, 69)

print(X_tactile_train.shape)

(4193, 6, 69)


In [13]:
tactile_mean = X_tactile_train.mean(axis=(0, 2), keepdims=True)
tactile_std = X_tactile_train.std(axis=(0, 2), keepdims=True)

print("Mean shape:", tactile_mean.shape)
print("Std shape:", tactile_std.shape)

Mean shape: (1, 6, 1)
Std shape: (1, 6, 1)


In [22]:
X_tactile_train = (
    X_tactile_train - tactile_mean
) / tactile_std

X_tactile_val = (
    X_tactile_val - tactile_mean
) / tactile_std

X_tactile_test = (
    X_tactile_test - tactile_mean
) / tactile_std

print("Train mean:", X_tactile_train.mean())
print("Train std:", X_tactile_train.std())

Train mean: -0.14116547
Train std: 0.12989849


In [24]:
feature_scaler = StandardScaler()

X_features_train = feature_scaler.fit_transform(
    X_features_train
)

X_features_val = feature_scaler.transform(
    X_features_val
)

X_features_test = feature_scaler.transform(
    X_features_test
)

print(
    "Training feature mean:",
    X_features_train.mean()
)

print(
    "Training feature std:",
    X_features_train.std()
)

Training feature mean: 1.7644171e-09
Training feature std: 1.0


In [25]:
y_train = y_train.to_numpy(dtype=np.int64)
y_val = y_val.to_numpy(dtype=np.int64)
y_test = y_test.to_numpy(dtype=np.int64)

print("X tactile train:", X_tactile_train.shape)
print("X features train:", X_features_train.shape)
print("y train:", y_train.shape)

print("X tactile val:", X_tactile_val.shape)
print("X features val:", X_features_val.shape)
print("y val:", y_val.shape)

print("X tactile test:", X_tactile_test.shape)
print("X features test:", X_features_test.shape)
print("y test:", y_test.shape)

X tactile train: (4193, 6, 69)
X features train: (4193, 33)
y train: (4193,)
X tactile val: (898, 6, 69)
X features val: (898, 33)
y val: (898,)
X tactile test: (899, 6, 69)
X features test: (899, 33)
y test: (899,)


In [26]:
# Final preprocessing verification

print("===== SHAPES =====")
print("Tactile train:", X_tactile_train.shape)
print("Tactile val:  ", X_tactile_val.shape)
print("Tactile test: ", X_tactile_test.shape)

print("\nFeatures train:", X_features_train.shape)
print("Features val:  ", X_features_val.shape)
print("Features test: ", X_features_test.shape)

print("\ny train:", y_train.shape)
print("y val:  ", y_val.shape)
print("y test: ", y_test.shape)


print("\n===== CLASS COUNTS =====")
print("Train:", np.bincount(y_train))
print("Val:  ", np.bincount(y_val))
print("Test: ", np.bincount(y_test))


print("\n===== NaN CHECK =====")
print("Tactile train NaN:", np.isnan(X_tactile_train).sum())
print("Features train NaN:", np.isnan(X_features_train).sum())

print("\n===== INFINITY CHECK =====")
print("Tactile train Inf:", np.isinf(X_tactile_train).sum())
print("Features train Inf:", np.isinf(X_features_train).sum())

===== SHAPES =====
Tactile train: (4193, 6, 69)
Tactile val:   (898, 6, 69)
Tactile test:  (899, 6, 69)

Features train: (4193, 33)
Features val:   (898, 33)
Features test:  (899, 33)

y train: (4193,)
y val:   (898,)
y test:  (899,)

===== CLASS COUNTS =====
Train: [559 512 491 509 542 540 508 532]
Val:   [119 110 105 109 116 116 109 114]
Test:  [120 110 105 109 117 116 108 114]

===== NaN CHECK =====
Tactile train NaN: 0
Features train NaN: 0

===== INFINITY CHECK =====
Tactile train Inf: 0
Features train Inf: 0


In [27]:
# ============================================
# SVM BASELINE PREPARATION
# ============================================

# Flatten tactile data
X_tactile_train_flat = X_tactile_train.reshape(X_tactile_train.shape[0], -1)
X_tactile_val_flat = X_tactile_val.reshape(X_tactile_val.shape[0], -1)
X_tactile_test_flat = X_tactile_test.reshape(X_tactile_test.shape[0], -1)

print("Tactile flattened:")
print("Train:", X_tactile_train_flat.shape)
print("Val:  ", X_tactile_val_flat.shape)
print("Test: ", X_tactile_test_flat.shape)

Tactile flattened:
Train: (4193, 414)
Val:   (898, 414)
Test:  (899, 414)


In [28]:
X_svm_train = np.concatenate(
    [X_tactile_train_flat, X_features_train],
    axis=1
)

X_svm_val = np.concatenate(
    [X_tactile_val_flat, X_features_val],
    axis=1
)

X_svm_test = np.concatenate(
    [X_tactile_test_flat, X_features_test],
    axis=1
)

print("\nSVM input:")
print("Train:", X_svm_train.shape)
print("Val:  ", X_svm_val.shape)
print("Test: ", X_svm_test.shape)


SVM input:
Train: (4193, 447)
Val:   (898, 447)
Test:  (899, 447)
